***
# Homework 8: HTML and JSON

*Course:** STAT 606 - Computing in Data Science and Statistics SP24

**Name:** Shrivats Sudhir

**NetID:** ssudhir2

**Email:** ssudhir2@wisc.edu

**Collaborators:** Samuel Merten, Amy Merkelz

**Date:** March 31st, 2024
***

In [1]:
from urllib.error import HTTPError
import urllib.request
from bs4 import BeautifulSoup

import pandas as pd

import json

## 1.) Warmup: Parsing HTML (spent $\approx$ 10 minutes)

**Let’s get started using `BeautifulSoup` with a couple of simple exercises. Both of the following subproblems ask you to retrieve the HTML from a URL given by an argument `s` and determine some simple information about that HTML.** 

**In both functions, you should raise an appropriate error in the event that `s` is not a string, and you should raise an `HTTPError` with an appropriate error message in the event that the success code that
results from trying to access the URL is not code 200.** 

**You may rely on `requests` and/or `urllib` to raise an error for you in the event that `s` is a string but does not encode a valid URL.**

**Write a function `get_page_title` that takes a string `s` as its only argument and returns a string.** 

**Your function should try to treat `s` as a URL, and return the string stored in the title tag of the HTML page stored at that URL.**

**In the unlikely event that the URL has more than one title tag, your function should return the text stored in the first one. If no such title exists, your function should return `None`.**

**A good way to test this function is to simply check that your function returns the string matching the title in your browser– the page title will always be displayed in the tab in which you have the page open.**

In [2]:
def get_page_title(s):
    
    '''
    Get the title of a webpage.

    Inputs:
    s: str
       A string representing a URL.

    Outputs:
    title: str
           The title of the webpage.
    '''

    if not isinstance(s, (str, )):
        raise TypeError(f'URL ({s}) must be a string.') # raises error if s is not a string
    
    try:
        response = urllib.request.urlopen(s) # uses urllib to open the URL
        if response.getcode() != 200:
            raise HTTPError(f'Could not open URL ({s}).') # raises error if the URL is not 200 status code
    except HTTPError as error:
        raise HTTPError(f'HTTP error occurred: {error}') # raises error if an HTTP error occurs
    
    html = response.read() # reads the HTML content of the URL
    soup = BeautifulSoup(html, 'html.parser') # uses BeautifulSoup to parse the HTML content
    title = soup.title.string # gets the title of the webpage

    return title

**Write a function `count_links` that takes a string `s` encoding a URL as its only argument and returns an integer corresponding to the number of hyperlinks on the webpage stored at that URL.** 

**The homework instructions page linked to above is an easy page to test your code on– there are only three links.**

In [3]:
def count_links(s):
    
    '''
    Count the number of links in a webpage.

    Inputs:
    s: str
       A string representing a URL.

    Outputs:
    num_links: int
               The number of links in the webpage.
    '''
    
    if not isinstance(s, (str, )):
        raise TypeError(f'URL ({s}) must be a string.') # raises error if s is not a string
    
    try:
        response = urllib.request.urlopen(s)
        if response.getcode() != 200:
            raise HTTPError(f'Could not open URL ({s}).') # raises error if the URL is not 200 status code
    except HTTPError as error:
        raise HTTPError(f'HTTP error occurred: {error}') # raises error if an HTTP error occurs
    
    html = response.read() # reads the HTML content of the URL
    soup = BeautifulSoup(html, 'html.parser') # uses BeautifulSoup to parse the HTML content
    num_links = len(soup.find_all('a')) # gets the number of links in the webpage by finding all 'a' tags
    
    return num_links

## 2.) Retrieving Data from the Web (7 points, spent $\approx$ 40 minutes)

**In this problem, we’ll scrape data from Wikipedia using `BeautifulSoup`. Documentation for `BeauitfulSoup` can be found at *https://www.crummy.com/software/BeautifulSoup/bs4/doc/*.** 

**As mentioned in lecture, there is another package, called requests, which is becoming quite popular, which you are welcome to use for this problem instead, if you wish. Documentation for the requests package can be found at *http://docs.python-requests.org/en/master/*.**

**Suppose you are trying to choose a city to vacation in. A major factor in your decision is weather. Conveniently, lots of weather information is present in the Wikipedia articles for most world cities.** 

**Your job in this problem is to use `BeautifulSoup` to retrieve weather information from Wikipedia articles. We should note that in practice, such information is more easily obtained from, for example, the National Oceanic and Atmospheric Administration (NOAA) in the case of American cities, and from analogous organizations in other countries.**

**Look at a few Wikipedia pages corresponding to cities. For example:**

* *https://en.wikipedia.org/wiki/Madison,_Wisconsin*

* *https://en.wikipedia.org/wiki/Buenos_Aires*

* *https://en.wikipedia.org/wiki/Harbin*

**Note that most city pages include a table titled something like “Climate data for [Cityname] (normals YYYY-YYYY, extremes YYYY-YYYY)” Find a Wikipedia page for a city that includes such a table (such as one of the three above).**

**In your jupyter notebook, open the URL and read the HTML using either `urllib` or `requests`, and parse it with `BeautifulSoup` using the standard parser, `html.parser`.**

**Have a look at the parsed HTML and find the climate data table, which will have the tag table and will contain a child tag `th` containing a string similar to**

`Climate data for [Cityname] (normals YYYY-YYYY, extremes YYYY-YYYY).`

**Find the node in the `BeautifulSoup` object corresponding to this table. What is the structure of this node of the tree (e.g., how many children does the table have, what are their tags, etc.)? You may want to learn a bit about the structure of HTML tables by looking at the resources available on these websites:**

* *https://developer.mozilla.org/en-US/docs/Web/HTML/Element/table*

* *https://www.w3schools.com/html/html_tables.asp*

* *https://www.w3.org/TR/html401/struct/tables.html*

After inspecting element for `s = 'https://en.wikipedia.org/wiki/Madison,_Wisconsin'`, I found the table for `Climate data for {City Name}` and pasted the CSS Selector and XPATH location:

CSS Selector: $\texttt{.mw-content-ltr > div:nth-child(110)}$

XPATH: $\texttt{/html/body/div[2]/div/div[3]/main/div[3]/div[3]/div[1]/div[16]}$

I also noticed the following:

* It is enclosed between `<div> <table> <tbody> ... </tbody> </table> </div>`

* The header is enclosed between `<tbody> <tr> ... </tr> </tbody>` and contains `<th colspan="14">`.

* `colspan="14"` always contains the following columns, (1.) Months, (2.) - (13.) Jan - Dec, (14.) Year.

**Write a function `retrieve_climate_table` that takes as its only argument a string representing a URL, and returns the `BeautifulSoup` tag object corresponding to the climate data table (if it exists in the page) and returns `None` if no such table exists on the page.**

**You should check that the URL is retrieved successfully, and raise an error if `urllib2` fails to successfully read the website.** 

**You may notice that some city pages include more than one climate data table or several nested tables (see, for example, https://en.wikipedia.org/wiki/Los_Angeles). In this case, your function may arbitrarily choose one of the tables to return as a BeautifulSoup object.**

In [4]:
def retrieve_climate_table(s):

    '''
    Retrieve the climate data table from a webpage.

    Inputs:
    s: str
       A string representing a URL.

    Outputs:
    table: bs4.element.Tag
           The HTML table containing the climate data. 
    '''
    
    if not isinstance(s, (str, )):
        raise TypeError(f'URL ({s}) must be a string.') # raises error if s is not a string
    
    try:
        response = urllib.request.urlopen(s) # uses urllib to open the URL
    except HTTPError as error:
        raise HTTPError(f'HTTP error occurred: {error}') # raises error if an HTTP error occurs
    
    html = response.read() # reads the HTML content of the URL
    soup = BeautifulSoup(html, 'html.parser') # uses BeautifulSoup to parse the HTML content

    for table in soup.find_all('table'): # finds all <tables> ... </tables> in the HTML content
        th = table.find_all('th') # finds all <th> ... </th> in the table
        for i in th:
            # iterates through each <th> tag
            # checks if <th> tag contains colspan="14"
            # attrs is a dictionary of the tag's attributes
            if 'colspan' in i.attrs.keys() and '14' in i.attrs.values():
                return table
    return None    

**As you look at some of the climate data tables, you may notice that different cities' tables contain different information. For example, not all cities include snowfall data.** 

**Write a function `list_climate_table_row_names` that takes as its only argument a Wikipedia URL and returns a list of the row names of the climate data table, or returns `None` if no such table exists. The list returned by your function should, ideally, consist solely of Python strings (either Unicode or ASCII), and should not include any BeautifulSoup objects or HTML, and the strings should not have any trailing whitespace (Hint: see the `BeautifulSoup` method get_text()).** 

**The list returned by your script should not include an entry corresponding to the `Climate data for...` row in the table.** 

**Second hint: you are looking for HTML table header (`th`) objects. The HTML attribute `scope` is your friend here, because in the context of an HTML table it tells you when a `th` tag is the header of a row or a column.**

In [5]:
def list_climate_table_row_names(s):

    '''
    List the row names of the climate data table.

    Inputs:
    s: str
       A string representing a URL.

    Outputs:
    row_names: list
               A list of strings representing the row names of the climate data table. 
    '''

    table_html = retrieve_climate_table(s) # retrieves the climate data HTML table from the URL

    if table_html is None: # if no table is found, return None
        return None
    
    row_names = [] # initializes an empty list to store the row names
    is_month = True # boolean variable to check if row is not corresponding to `Climate data for ...`
    for row in table_html.find_all('th'): # finds all <th> ... </th> in the table
        # iterates through each <th> tag
        # checks if <th> tag contains 'scope' and 'row' in its attributes
        # attrs is a dictionary of the tag's attributes
        if 'scope' in row.attrs.keys() and 'row' in row.attrs.values():
            if is_month: # if row is corresponding to `Climate data for ...`, set is_month to False and skip iteration
                is_month = False
                next
            else:
                row_names.append(row.text.strip()) # appends the row name to the list of row names
    return row_names

**The next natural step would be to write a function that takes a URL and a row name and retrieves the data from that row of the climate data table (if the table exists and has that row name). Doing this would require some complicated string wrangling to get right, so I’ll spare you the trouble. Instead, please briefly describe either in pseudo code or in plain English how you would accomplish this, using the two functions you wrote above and the tools available to you in the `BeautifulSoup` package.** 

**Note: just to be clear, you do not have to write any code for this last step. Of course, if you want a challenge, you are welcome to try writing this code, but it is not required for this assignment**

I have abstracted the function described above so that the only input required is the URL string `s`. My function does the following:

* Gathers all the row names by running the `list_climate_table_row_names()` written above.

* Gathers all column elements (for each row name) inside a list called `elements`. Note that, I found that after inspecting element, each cell value is of the form `<td> number <br> (number) </td>`.

* Finally, I know that each splices of 13 numbers are the row data for each row name (associated for (1.) - (12.) Month names and (13.) Year). Thus, we initialize a dictionary with keys `'Month'` and values `'Jan', 'Feb',...,'Dec'` and `'Years'`, append row names as keys and its corresponding 13 elements as values, and return the `pd.DataFrame` associated to the dictionary.

To answer the above question, where the function requires both input url `s` as well as a row name (say `row_name`), the idea is that we use `find_all('td')` under the specific `find_all('th')` for `row_name` and gather all cell values up until the loop reaches the cell value where `col.attrs.values()` contains `border-left-width:medium`. That indicates the end of all column values for `row_name` as it is always attributed to the cell value for `row_name` corresponding to `Years`.

In [6]:
def list_column_data_rows(s):

    '''
    List the column data rows of the climate data table.

    Inputs:
    s: str
       A string representing a URL.

    Outputs:
    climate_data: pandas.DataFrame
                  A DataFrame containing the climate data.
    '''

    row_names = list_climate_table_row_names(s) # retrieves the row names of the climate data table from the URL
    table_html = retrieve_climate_table(s) # retrieves the climate data HTML table from the URL
    
    if (row_names is None) or (table_html is None): # if no row names or tables are found, return None
        return None
    
    else:

        elements = [] # initializes an empty list to store the elements of the table  
        for row in table_html.find_all('th'): # finds all <th> ... </th> in the table
            for col in table_html.find_all('td'): # finds all <td> ... </td> in the table corresponding to the <th> tag
                if row.text.strip() in row_names: # checks if the <th> tag is in the list of row names
                    elements.append(col.text.strip()) # appends the element to the list of elements

        # initializes a dictionary with Month key and Jan,...,Dec and Year values to store the data
        data = dict({'Month':['Jan', # 0
                              'Feb', # 1
                              'Mar', # 2
                              'Apr', # 3
                              'May', # 4
                              'Jun', # 5
                              'Jul', # 6
                              'Aug', # 7
                              'Sep', # 8 
                              'Oct', # 9
                              'Nov', # 10
                              'Dec', # 11
                              'Year']}) # 12

        for i in range(len(row_names)): 
            # iterates len(row_names) times
            # appends a splice of 13 consecutive elements to the dictionary
            # corresponds to Jan,...,Dec and Year values
            data[row_names[i]] = elements[i * 13 : (i+1) * 13]
        
        climate_data = pd.DataFrame(data) # creates a DataFrame from the dictionary, contains the climate data

    return climate_data

In [7]:
s = 'https://en.wikipedia.org/wiki/Madison,_Wisconsin'
list_column_data_rows(s).T

,0,1,2,3,4,5,6,7,8,9,10,11,12
Month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Year
Record high °F (°C),58(14),70(21),83(28),94(34),101(38),101(38),107(42),102(39),99(37),90(32),77(25),68(20),107(42)
Mean maximum °F (°C),46.2(7.9),51.3(10.7),67.1(19.5),79.1(26.2),85.6(29.8),91.0(32.8),92.2(33.4),90.4(32.4),87.6(30.9),79.4(26.3),63.9(17.7),50.8(10.4),94.1(34.5)
Mean daily maximum °F (°C),27.0(−2.8),31.2(−0.4),43.6(6.4),56.9(13.8),69.0(20.6),78.6(25.9),82.1(27.8),79.9(26.6),72.9(22.7),59.6(15.3),44.8(7.1),32.3(0.2),56.5(13.6)
Daily mean °F (°C),19.4(−7.0),23.0(−5.0),34.4(1.3),46.3(7.9),58.1(14.5),68.0(20.0),71.9(22.2),69.7(20.9),62.0(16.7),49.7(9.8),36.7(2.6),25.3(−3.7),47.0(8.3)
Mean daily minimum °F (°C),11.8(−11.2),14.9(−9.5),25.1(−3.8),35.8(2.1),47.1(8.4),57.4(14.1),61.6(16.4),59.5(15.3),51.0(10.6),39.8(4.3),28.7(−1.8),18.2(−7.7),37.6(3.1)
Mean minimum °F (°C),−10.6(−23.7),−5.5(−20.8),4.2(−15.4),21.3(−5.9),32.1(0.1),43.2(6.2),49.9(9.9),48.1(8.9),35.8(2.1),25.3(−3.7),12.2(−11.0),−2.6(−19.2),−13.9(−25.5)
Record low °F (°C),−37(−38),−29(−34),−29(−34),0(−18),19(−7),31(−1),36(2),35(2),25(−4),12(−11),−14(−26),−28(−33),−37(−38)
Average precipitation inches (mm),1.47(37),1.52(39),2.26(57),3.78(96),4.10(104),5.28(134),4.51(115),4.16(106),3.43(87),2.77(70),2.22(56),1.63(41),37.13(943)
Average snowfall inches (cm),13.7(35),12.8(33),7.0(18),2.6(6.6),0.1(0.25),0.0(0.0),0.0(0.0),0.0(0.0),0.0(0.0),0.6(1.5),3.0(7.6),12.0(30),51.8(132)


## 3.) JSON (4 points, spent $\approx$ 7 minutes)

**In this problem, you’ll get a bit of practice working with JSON objects.**

**Download yelp_simplified.json from http://pages.stat.wisc.edu/~kdlevin/teaching/Spring2024/STAT606/yelp_simplified.json. This file contains a string representation of a JSON object generated by the Yelp API when I asked to retrieve all restaurants matching the search term ’coffee’ within 200 meters of the UW-Madison statistics department. I’ve removed some of the attributes for the sake of simplicity.** 

**We’ll discuss how to interact with APIs like the Yelp API and others later in the course. Load the JSON string into a Python json object called `yelp_json`. Please include `yelp_simplified.json` in your submission.** 

**Hint: you can read a JSON object directly from the file by creating a file handle `f` and writing `json.load(f)`. Don’t forget to close the file after you’re done reading from it**

In [88]:
with open('yelp_simplified.json', 'r') as f: # opens the JSON file, reads the content, and closes the file
    yelp_json = json.load(f) # loads the JSON content into a dictionary

yelp_json

{'businesses': [{'name': "Aldo's Cafe",
   'review_count': 31,
   'categories': [{'alias': 'cafes', 'title': 'Cafes'},
    {'alias': 'sandwiches', 'title': 'Sandwiches'}],
   'rating': 3.5,
   'coordinates': {'latitude': 43.073180718899,
    'longitude': -89.4076565447393},
   'price': '$',
   'phone': '+16082043943',
   'distance': 81.31877892848861},
  {'name': 'The Library Cafe & Bar',
   'review_count': 48,
   'categories': [{'alias': 'coffee', 'title': 'Coffee & Tea'},
    {'alias': 'lounges', 'title': 'Lounges'},
    {'alias': 'tradamerican', 'title': 'American (Traditional)'}],
   'rating': 3.5,
   'coordinates': {'latitude': 43.0730400085449,
    'longitude': -89.4092178344727},
   'price': '$$',
   'phone': '+16082511200',
   'distance': 171.28210299650746},
  {'name': 'Badger Market',
   'review_count': 2,
   'categories': [{'alias': 'breakfast_brunch', 'title': 'Breakfast & Brunch'},
    {'alias': 'juicebars', 'title': 'Juice Bars & Smoothies'},
    {'alias': 'cafes', 'title

**The `'businesses'` attribute of the JSON attribute has as its value an array whose elements are themselves JSON objects, each of which represents one of the three establishments within 200 meters of the Medical Sciences Center matching my search for `'coffee'`. Extract this array (you can simply treat it as a Python list!) and save it in a variable called `search_results`.**

In [89]:
search_results = yelp_json['businesses'] # retrieves the 'businesses' key (attribute) from the dictionary
search_results

[{'name': "Aldo's Cafe",
  'review_count': 31,
  'categories': [{'alias': 'cafes', 'title': 'Cafes'},
   {'alias': 'sandwiches', 'title': 'Sandwiches'}],
  'rating': 3.5,
  'coordinates': {'latitude': 43.073180718899, 'longitude': -89.4076565447393},
  'price': '$',
  'phone': '+16082043943',
  'distance': 81.31877892848861},
 {'name': 'The Library Cafe & Bar',
  'review_count': 48,
  'categories': [{'alias': 'coffee', 'title': 'Coffee & Tea'},
   {'alias': 'lounges', 'title': 'Lounges'},
   {'alias': 'tradamerican', 'title': 'American (Traditional)'}],
  'rating': 3.5,
  'coordinates': {'latitude': 43.0730400085449,
   'longitude': -89.4092178344727},
  'price': '$$',
  'phone': '+16082511200',
  'distance': 171.28210299650746},
 {'name': 'Badger Market',
  'review_count': 2,
  'categories': [{'alias': 'breakfast_brunch', 'title': 'Breakfast & Brunch'},
   {'alias': 'juicebars', 'title': 'Juice Bars & Smoothies'},
   {'alias': 'cafes', 'title': 'Cafes'}],
  'rating': 2.5,
  'coordinat

**Pick out one of the JSON elements from the list `search_results` and examine its attributes. Extract the attributes and save them in a Python list `resto_attrs_sorted`, with the attributes sorted in non-decreasing order.** 

**Hint: the attributes of a JSON object in the Python json module really are just the keys of a dictionary, so all you need to do is extract the keys of a dictionary, save them in a list, and sort that list.**

In [90]:
resto_attrs_sorted = list(sorted(search_results[0].keys())) # retrieves the keys (attributes) of the first restaurant and sorts them
resto_attrs_sorted

['categories',
 'coordinates',
 'distance',
 'name',
 'phone',
 'price',
 'rating',
 'review_count']

**A few of us in the statistics department have decided to follow our passion for baking and open the George E. P. Box Bakery in the basement of MSC.** 

**Create a Python dictionary `gepbox_json` representing our bakery’s JSON object. It should have the same structure as the other restaurants in the JSON search results. In particular:**

* **The name of our bakery on Yelp will be `'G. E. P. Box Bakery'`.**

* **Since our bakery is brand new, let us set its `'review_count'` to 0.**

* **The `'categories'` attribute should be an array containing a single JSON object `{’alias’: ’cafes’, ’title’: ’Cafes’}`**

* **Set the `'rating'` attribute to have value 5.0.**

* **The coordinates of the MSC are latitude `43.074281` and longitude `-89.407391`.**

* **We aim to provide affordable croissants to statisticians of all income levels. The `'price'` attribute should be a single dollar sign.**

* **The phone number will be the same as the department office: `+16082622598`.**

* **The other restaurants in the search results have a `'distance'` attribute, which is the distance from MSC. Since our bakery is in the basement of MSC, you can set the `'distance'` attribute to have value 0.0.**

In [91]:
# Creates new dictionary in the format of the restaurant
gepbox_json = dict({'name': 'G. E. P. Box Bakery', # name of the restaurant 
                   'review_count': 0, # number of reviews
                   'categories': [{'alias': 'cafes', 'title': 'Cafes'}], # array containing single JSON object with alias and title
                   'rating': 5.0, # rating of the restaurant
                   'coordinates': {'latitude': 43.074281, 'longitude': -89.407391}, # latitude and longitude of the restaurant
                   'price': '$', # price level of the restaurant
                   'phone': '+16082622598', # phone number of the restaurant
                   'distance': 0.0}) # distance of the restaurant
gepbox_json

{'name': 'G. E. P. Box Bakery',
 'review_count': 0,
 'categories': [{'alias': 'cafes', 'title': 'Cafes'}],
 'rating': 5.0,
 'coordinates': {'latitude': 43.074281, 'longitude': -89.407391},
 'price': '$',
 'phone': '+16082622598',
 'distance': 0.0}

**Use the Python json module to create a string representation of our `gepbox_json` object, and save it in a file `gepbox.json`. Please include this file in your submission.** 

**Hint: to make sure that you have created your JSON file correctly, try reading it back into Python using the same pattern you used above to read in `yelp_simplified.json`.**

In [92]:
with open('gepbox.json', 'w') as f: # opens the JSON file, writes the content, and closes the file
    json.dump(gepbox_json, f) # writes the JSON content from gepbox_json to the file